# 最終的に完成したもの

In [ ]:
!git clone https://github.com/ryy1210/RMT_utils
import sys
sys.path.append("/content/RMT_utils")

import funcs1

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.optimize as opt
# ==========================================
# 動作検証用シミュレーション
# ==========================================
def run_simulation():
    # 1. パラメータの設定
    m, n = 1000, 5000  # (m <= n)
    gamma = m / n      # Q = n/m の逆数

    print(f"行列サイズ: {m} x {n}")

    # 異分散（ヘテロスケダスティック）構造の生成
    #np.random.seed(42)
    # Sigma_diag = np.random.uniform(0.1, 5.0, m)
    # Omega_diag = np.random.uniform(0.1, 5.0, n)

    Sigma_diag = np.random.exponential(1, m)
    Omega_diag = np.random.exponential(1, n)


    # Y ~ N(0, Omega \otimes Sigma) の生成
    Z = np.random.randn(m, n)
    Y = Z * np.sqrt(Sigma_diag[:, None]) * np.sqrt(Omega_diag[None, :])

    print("オリジナルの固有値を計算中...")
    X_orig = Y @ Y.T / n
    evals_orig = np.linalg.eigvalsh(X_orig)

    # ==========================================
    # Algorithm 1 の適用
    # ==========================================
    print("Algorithm 1 (Dyson Equalizer) を適用中...")
    Y_hat, x_hat, y_hat = funcs1.dyson_equalizer_algorithm1(Y)

    # 補正後のESD計算
    X_hat = Y_hat @ Y_hat.T / n
    evals_hat = np.linalg.eigvalsh(X_hat)

    # ---------------------------------------------------------
    # MP分布の理論曲線の計算
    # ---------------------------------------------------------
    lambda_plus = (1 + np.sqrt(gamma))**2
    lambda_minus = (1 - np.sqrt(gamma))**2
    x_ax = np.linspace(max(0, lambda_minus - 0.5), lambda_plus + 0.5, 1000)

    valid_mask = (x_ax >= lambda_minus) & (x_ax <= lambda_plus)
    rho_mp = np.zeros_like(x_ax)
    rho_mp[valid_mask] = np.sqrt((lambda_plus - x_ax[valid_mask]) * (x_ax[valid_mask] - lambda_minus)) / (2 * np.pi * gamma * x_ax[valid_mask])

    result_orig = funcs1.bema_algorithm1_from_data(Y, alpha=0.2, beta=0.1)

    print("BEMA Algorithm 1")
    print(f"sigma^2_hat = {result_orig['sigma2_hat']:.4f}")
    print(f"s_hat        = {result_orig['s_hat']}")
    print(f"threshold    = {result_orig['threshold']:.4f}")


    result_hat = funcs1.bema_algorithm1_from_data(Y_hat, alpha=0.2, beta=0.1)

    print("After Dyson Equalizer")
    print(f"sigma^2_hat = {result_hat['sigma2_hat']:.4f}")
    print(f"s_hat        = {result_hat['s_hat']}")
    print(f"threshold    = {result_hat['threshold']:.4f}")

    sigma2_bema = result_hat['sigma2_hat']
    lambda_plus_bema = sigma2_bema * (1 + np.sqrt(gamma))**2
    lambda_minus = sigma2_bema * (1 - np.sqrt(gamma))**2



    # BEMAが推定したMP曲線の描画
    lambda_minus = sigma2_bema * (1 - np.sqrt(gamma))**2
    x = np.linspace(lambda_minus, lambda_plus_bema, 500)
    rho_mp_bema = np.sqrt((lambda_plus_bema - x) * (x - lambda_minus)) / (2 * np.pi * gamma * x * sigma2_bema)


    # ---------------------------------------------------------
    # プロット
    # ---------------------------------------------------------
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    bins = 100

    axes[0].hist(evals_orig, bins=bins, density=True, color='salmon', alpha=0.8)
    axes[0].plot(x, rho_mp_bema, 'k--', lw=2, label='Theoretical MP Law')
    axes[0].set_title('Original Heteroscedastic ESD', fontsize=14)
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    axes[1].hist(evals_hat, bins=bins, density=True, color='dodgerblue', alpha=0.8)
    axes[1].plot(x_ax, rho_mp, 'k--', lw=2, label='Theoretical MP Law')
    axes[1].set_title('Algorithm 1: Dyson Equalizer Corrected ESD', fontsize=14)
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()
    print("完了しました。")

if __name__ == "__main__":
    run_simulation()

# 未完成のときのもの

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def dyson_equalizer_algorithm1(Y):
    """
    Landa & Kluger (2024) - Algorithm 1: The Dyson Equalizer
    論文の数式と記法に完全に対応させた実装。

    Input:
        Y: Data matrix (m x n), m <= n
    Returns:
        Y_hat: Normalized data matrix
        x_hat: Row scaling vector
        y_hat: Column scaling vector
    """
    m, n = Y.shape
    if m > n:
        raise ValueError("Input matrix Y must have m <= n. Transpose Y if necessary.")

    # 1: Compute the SVD of Y
    # U: m x m, sigma: m, V_h: n x n
    U, sigma, V_h = np.linalg.svd(Y, full_matrices=True)
    V = V_h.T  # V \in R^{n x n} (右特異ベクトルを列に持つ行列)

    # 2: Set eta as the median singular value of Y
    eta = np.median(sigma)

    # 3: Compute the vectors g_hat^(1) and g_hat^(2)
    # 論文 (3) 式の計算（行列演算で高速化）
    term1 = eta / (sigma**2 + eta**2)
    term2 = term1 - (1 / eta)

    # U は m x m, sigma は要素数 m
    g1_hat = (U**2) @ term1

    # V は n x n. sum は k=1 から m までなので V の最初の m 列を使用
    g2_hat = (1 / eta) + (V[:, :m]**2) @ term2

    # 4: Compute the vectors x_hat and y_hat
    # L1ノルム ||g_hat^(1)||_1 と ||g_hat^(2)||_1 の計算
    g1_norm1 = np.sum(np.abs(g1_hat))
    g2_norm1 = np.sum(np.abs(g2_hat))

    # 論文 (4) 式の計算
    x_hat = (1 / np.sqrt(m - eta * g1_norm1)) * ((1 / g1_hat) - eta)
    y_hat = (1 / np.sqrt(n - eta * g2_norm1)) * ((1 / g2_hat) - eta)

    # 数値的安定性のための安全策（負値の平方根エラー回避）
    x_hat = np.maximum(1e-12, x_hat)
    y_hat = np.maximum(1e-12, y_hat)

    # 5: Form the normalized data matrix Y_hat
    # Y_hat = (D_{x_hat})^{-1/2} Y (D_{y_hat})^{-1/2}
    Y_hat = Y / (np.sqrt(x_hat[:, None]) * np.sqrt(y_hat[None, :]))

    return Y_hat, x_hat, y_hat

# ==========================================
# 動作検証用シミュレーション
# ==========================================
def run_simulation():
    # 1. パラメータの設定
    m, n = 1000, 5000  # (m <= n)
    gamma = m / n      # Q = n/m の逆数

    print(f"行列サイズ: {m} x {n}")

    # 異分散（ヘテロスケダスティック）構造の生成
    #np.random.seed(42)
    # Sigma_diag = np.random.uniform(0.1, 5.0, m)
    # Omega_diag = np.random.uniform(0.1, 5.0, n)

    Sigma_diag = np.random.exponential(1, m)
    Omega_diag = np.random.exponential(1, n)


    # Y ~ N(0, Omega \otimes Sigma) の生成
    Z = np.random.randn(m, n)
    Y = Z * np.sqrt(Sigma_diag[:, None]) * np.sqrt(Omega_diag[None, :])

    print("オリジナルの固有値を計算中...")
    X_orig = Y @ Y.T / n
    evals_orig = np.linalg.eigvalsh(X_orig)

    # ==========================================
    # Algorithm 1 の適用
    # ==========================================
    print("Algorithm 1 (Dyson Equalizer) を適用中...")
    Y_hat, x_hat, y_hat = dyson_equalizer_algorithm1(Y)

    # 補正後のESD計算
    X_hat = Y_hat @ Y_hat.T / n
    evals_hat = np.linalg.eigvalsh(X_hat)

    # ---------------------------------------------------------
    # MP分布の理論曲線の計算
    # ---------------------------------------------------------
    lambda_plus = (1 + np.sqrt(gamma))**2
    lambda_minus = (1 - np.sqrt(gamma))**2
    x_ax = np.linspace(max(0, lambda_minus - 0.5), lambda_plus + 0.5, 1000)

    valid_mask = (x_ax >= lambda_minus) & (x_ax <= lambda_plus)
    rho_mp = np.zeros_like(x_ax)
    rho_mp[valid_mask] = np.sqrt((lambda_plus - x_ax[valid_mask]) * (x_ax[valid_mask] - lambda_minus)) / (2 * np.pi * gamma * x_ax[valid_mask])

    # ---------------------------------------------------------
    # プロット
    # ---------------------------------------------------------
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    bins = 100

    axes[0].hist(evals_orig, bins=bins, density=True, color='salmon', alpha=0.8)
    axes[0].plot(x_ax, rho_mp, 'k--', lw=2, label='Theoretical MP Law')
    axes[0].set_title('Original Heteroscedastic ESD', fontsize=14)
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    axes[1].hist(evals_hat, bins=bins, density=True, color='dodgerblue', alpha=0.8)
    axes[1].plot(x_ax, rho_mp, 'k--', lw=2, label='Theoretical MP Law')
    axes[1].set_title('Algorithm 1: Dyson Equalizer Corrected ESD', fontsize=14)
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()
    print("完了しました。")

if __name__ == "__main__":
    run_simulation()

BEMAによる分散の分散の推定（BEMA著者のgithub参照．モンテカルロシミュレーションによる実装？）

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.optimize as opt

def dyson_equalizer_algorithm1(Y):
    """
    Landa & Kluger (2024) - Algorithm 1: The Dyson Equalizer
    論文の数式と記法に完全に対応させた実装。

    Input:
        Y: Data matrix (m x n), m <= n
    Returns:
        Y_hat: Normalized data matrix
        x_hat: Row scaling vector
        y_hat: Column scaling vector
    """
    m, n = Y.shape
    if m > n:
        raise ValueError("Input matrix Y must have m <= n. Transpose Y if necessary.")

    # 1: Compute the SVD of Y
    # U: m x m, sigma: m, V_h: n x n
    U, sigma, V_h = np.linalg.svd(Y, full_matrices=True)
    V = V_h.T  # V \in R^{n x n} (右特異ベクトルを列に持つ行列)

    # 2: Set eta as the median singular value of Y
    eta = np.median(sigma)

    # 3: Compute the vectors g_hat^(1) and g_hat^(2)
    # 論文 (3) 式の計算（行列演算で高速化）
    term1 = eta / (sigma**2 + eta**2)
    term2 = term1 - (1 / eta)

    # U は m x m, sigma は要素数 m
    g1_hat = (U**2) @ term1

    # V は n x n. sum は k=1 から m までなので V の最初の m 列を使用
    g2_hat = (1 / eta) + (V[:, :m]**2) @ term2

    # 4: Compute the vectors x_hat and y_hat
    # L1ノルム ||g_hat^(1)||_1 と ||g_hat^(2)||_1 の計算
    g1_norm1 = np.sum(np.abs(g1_hat))
    g2_norm1 = np.sum(np.abs(g2_hat))

    # 論文 (4) 式の計算
    x_hat = (1 / np.sqrt(m - eta * g1_norm1)) * ((1 / g1_hat) - eta)
    y_hat = (1 / np.sqrt(n - eta * g2_norm1)) * ((1 / g2_hat) - eta)

    # 数値的安定性のための安全策（負値の平方根エラー回避）
    x_hat = np.maximum(1e-12, x_hat)
    y_hat = np.maximum(1e-12, y_hat)

    # 5: Form the normalized data matrix Y_hat
    # Y_hat = (D_{x_hat})^{-1/2} Y (D_{y_hat})^{-1/2}
    Y_hat = Y / (np.sqrt(x_hat[:, None]) * np.sqrt(y_hat[None, :]))

    return Y_hat, x_hat, y_hat

# ==========================================
# 動作検証用シミュレーション
# ==========================================
def run_simulation():
    # 1. パラメータの設定
    m, n = 1000, 5000  # (m <= n)
    gamma = m / n      # Q = n/m の逆数

    print(f"行列サイズ: {m} x {n}")

    # 異分散（ヘテロスケダスティック）構造の生成
    #np.random.seed(42)
    # Sigma_diag = np.random.uniform(0.1, 5.0, m)
    # Omega_diag = np.random.uniform(0.1, 5.0, n)

    Sigma_diag = np.random.exponential(1, m)
    Omega_diag = np.random.exponential(1, n)


    # Y ~ N(0, Omega \otimes Sigma) の生成
    Z = np.random.randn(m, n)
    Y = Z * np.sqrt(Sigma_diag[:, None]) * np.sqrt(Omega_diag[None, :])

    print("オリジナルの固有値を計算中...")
    X_orig = Y @ Y.T / n
    evals_orig = np.linalg.eigvalsh(X_orig)

    # ==========================================
    # Algorithm 1 の適用
    # ==========================================
    print("Algorithm 1 (Dyson Equalizer) を適用中...")
    Y_hat, x_hat, y_hat = dyson_equalizer_algorithm1(Y)

    # 補正後のESD計算
    X_hat = Y_hat @ Y_hat.T / n
    evals_hat = np.linalg.eigvalsh(X_hat)

    # ---------------------------------------------------------
    # MP分布の理論曲線の計算
    # ---------------------------------------------------------
    lambda_plus = (1 + np.sqrt(gamma))**2
    lambda_minus = (1 - np.sqrt(gamma))**2
    x_ax = np.linspace(max(0, lambda_minus - 0.5), lambda_plus + 0.5, 1000)

    valid_mask = (x_ax >= lambda_minus) & (x_ax <= lambda_plus)
    rho_mp = np.zeros_like(x_ax)
    rho_mp[valid_mask] = np.sqrt((lambda_plus - x_ax[valid_mask]) * (x_ax[valid_mask] - lambda_minus)) / (2 * np.pi * gamma * x_ax[valid_mask])

    def bema_loss(sigma2_proposal, evals_emp, gamma, p, alpha):
        """
        BEMAの損失関数 (BEMA.Rの `loss` 関数に相当)
        提案された分散 sigma^2 に基づいてMP分布に従うランダム行列をシミュレートし、
        経験的固有値のバルク部分（分位数）との二乗誤差を計算します。
        """
        n = int(p / gamma)
        L = np.zeros((10, p))

        # 提案された分散でランダム行列を10回モンテカルロシミュレーション
        for i in range(10):
            Z_sim = np.random.randn(p, n) * np.sqrt(sigma2_proposal)
            if p <= n:
                S_sim = Z_sim @ Z_sim.T / n
            else:
                S_sim = Z_sim.T @ Z_sim / n
            L[i, :] = np.sort(np.linalg.eigvalsh(S_sim))[::-1]

        evals_sim_mean = np.mean(L, axis=0)

        # alpha に基づいて、分布の「端（スパイクや微小固有値）」を切り落とす
        # 例: alpha=0.2 の場合、上位20%と下位20%を無視し、中間の60%のバルクだけで比較する
        idx_start = int(min(p, n) * alpha)
        idx_end = int(min(p, n) * (1 - alpha))

        evals_emp_bulk = evals_emp[idx_start:idx_end]
        evals_sim_bulk = evals_sim_mean[idx_start:idx_end]

        # バルク部分の分位数の二乗誤差
        loss = np.sum((evals_emp_bulk - evals_sim_bulk)**2)
        return loss

    def apply_bema(evals_emp, gamma, p, alpha=0.2):
        """
        BEMAアルゴリズムを実行し、真の分散 sigma^2 を推定します。
        """
        print("BEMAによる分散推定を実行中...")
        # scipy.optimize.minimize_scalar を用いて、損失関数を最小化する分散を探索
        res = opt.minimize_scalar(
            bema_loss,
            args=(evals_emp, gamma, p, alpha),
            bounds=(0.01, 10.0),
            method='bounded'
        )
        return res.x

    # 3. BEMAによるロバストな分散推定
    sigma2_bema = apply_bema(evals_orig, gamma, m, alpha=0.2)

    print(f"BEMAによる分散推定:   {sigma2_bema:.4f}")

    sigma2_bema_DE = apply_bema(evals_hat, gamma, m, alpha=0.2)

    print(f"DE後のBEMAによる分散推定:   {sigma2_bema_DE:.4f}")



    # BEMAが推定したMP曲線の描画
    lambda_plus_bema = sigma2_bema * (1 + np.sqrt(gamma))**2
    lambda_minus = sigma2_bema * (1 - np.sqrt(gamma))**2
    x = np.linspace(lambda_minus, lambda_plus_bema, 500)
    rho_mp_bema = np.sqrt((lambda_plus_bema - x) * (x - lambda_minus)) / (2 * np.pi * gamma * x * sigma2_bema)


    # ---------------------------------------------------------
    # プロット
    # ---------------------------------------------------------
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    bins = 100

    axes[0].hist(evals_orig, bins=bins, density=True, color='salmon', alpha=0.8)
    axes[0].plot(x, rho_mp_bema, 'k--', lw=2, label='Theoretical MP Law')
    axes[0].set_title('Original Heteroscedastic ESD', fontsize=14)
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    axes[1].hist(evals_hat, bins=bins, density=True, color='dodgerblue', alpha=0.8)
    axes[1].plot(x_ax, rho_mp, 'k--', lw=2, label='Theoretical MP Law')
    axes[1].set_title('Algorithm 1: Dyson Equalizer Corrected ESD', fontsize=14)
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()
    print("完了しました。")

if __name__ == "__main__":
    run_simulation()

DE後の分散が１になっていないので，BEMAがうまく実装できていなそう

In [ ]:
import numpy as np
import scipy.interpolate as interp
from scipy.stats import norm


def tw1_quantile(beta=0.1):
    """
    Type-I Tracy-Widom 分布の (1-beta) 分位点を返す。

    scipy に tracywidom がある環境ではそれを使用。
    ない場合は代表的な近似値を使う。
    """
    try:
        from scipy.stats import tracywidom
        return tracywidom.ppf(1 - beta, beta=1)
    except Exception:
        # Type-I Tracy-Widom TW1 の代表的な分位点近似
        # beta は右側確率。つまり返すのは 1-beta quantile。
        table = {
            0.20: -0.165,
            0.10:  0.450,
            0.05:  0.979,
            0.025: 1.454,
            0.01:  2.023,
            0.001: 3.272,
        }

        if beta in table:
            return table[beta]

        # 近い値を線形補間
        betas = np.array(sorted(table.keys()))
        vals = np.array([table[b] for b in betas])

        if beta < betas.min():
            return vals[0]
        if beta > betas.max():
            return vals[-1]

        return np.interp(beta, betas, vals)


def mp_pdf_zero_excluded(x, gamma, sigma2=1.0):
    """
    zero-excluded Marchenko-Pastur density.

    gamma = p / n.
    sigma2 = 1 のとき標準MP分布。

    gamma > 1 の場合、p x p sample covariance にはゼロ固有値が出るので、
    非ゼロ固有値に条件づけた zero-excluded density を使う。
    """
    x = np.asarray(x)

    a = sigma2 * (1 - np.sqrt(gamma)) ** 2
    b = sigma2 * (1 + np.sqrt(gamma)) ** 2

    pdf = np.zeros_like(x, dtype=float)

    mask = (x > a) & (x < b)
    xm = x[mask]

    # classical MP density の正規化係数は 2*pi*gamma*sigma2*x
    # gamma > 1 では非ゼロ部分の質量が 1/gamma なので、
    # zero-excluded にするため gamma 倍する。
    denom_gamma = min(gamma, 1.0)

    pdf[mask] = (
        np.sqrt((b - xm) * (xm - a))
        / (2 * np.pi * denom_gamma * sigma2 * xm)
    )

    return pdf


def mp_upper_quantiles(gamma, p_tilde, k_indices, grid_size=200000):
    """
    sigma2=1 の zero-excluded MP 分布について、
    k/p_tilde upper-quantile q_k を返す。

    k_indices は 1始まりの index を想定。
    """
    a = (1 - np.sqrt(gamma)) ** 2
    b = (1 + np.sqrt(gamma)) ** 2

    eps = 1e-10
    x_grid = np.linspace(a + eps, b - eps, grid_size)

    pdf = mp_pdf_zero_excluded(x_grid, gamma, sigma2=1.0)

    # 数値誤差補正のため、台形積分でCDFを作って正規化
    dx = x_grid[1] - x_grid[0]
    cdf = np.cumsum(pdf) * dx
    cdf = cdf / cdf[-1]

    # upper tail probability y = k / p_tilde
    # F(q_k) = 1 - y
    y_upper = np.asarray(k_indices, dtype=float) / p_tilde
    cdf_targets = 1.0 - y_upper

    inv_cdf = interp.interp1d(
        cdf,
        x_grid,
        bounds_error=False,
        fill_value=(a, b)
    )

    q = inv_cdf(cdf_targets)
    return q


def bema_algorithm1_from_eigenvalues(evals, p, n, alpha=0.2, beta=0.1):
    """
    BEMA Algorithm 1 for the standard spiked covariance model.

    Parameters
    ----------
    evals : array-like
        sample covariance matrix の非ゼロ固有値。
        降順でなくてもよい。
    p : int
        次元。Y が p x n のデータ行列なら p = Y.shape[0]。
    n : int
        サンプルサイズ。Y が p x n のデータ行列なら n = Y.shape[1]。
    alpha : float
        bulk eigenvalues の中央部分を選ぶパラメータ。
        論文のデフォルトは 0.2。
    beta : float
        over-estimation probability を制御するパラメータ。
        論文の実用デフォルトは 0.1。

    Returns
    -------
    result : dict
        sigma2_hat, K_hat, threshold, q_bulk, evals_sorted など。
    """
    evals = np.asarray(evals, dtype=float)
    evals = evals[evals > 1e-14]
    evals_sorted = np.sort(evals)[::-1]

    p_tilde = min(p, n)

    if len(evals_sorted) != p_tilde:
        # 数値的にゼロ固有値を除いた数がずれる場合に合わせる
        p_tilde = len(evals_sorted)

    gamma = p / n

    # 論文の index は 1 <= k <= p_tilde
    k_start = int(np.ceil(alpha * p_tilde))
    k_end = int(np.floor((1 - alpha) * p_tilde))

    # 1始まり index
    k_indices = np.arange(k_start, k_end + 1)

    # Python の配列 index は 0始まりなので -1
    evals_bulk = evals_sorted[k_indices - 1]

    q_bulk = mp_upper_quantiles(
        gamma=gamma,
        p_tilde=p_tilde,
        k_indices=k_indices
    )

    sigma2_hat = np.sum(q_bulk * evals_bulk) / np.sum(q_bulk ** 2)

    t_tw = tw1_quantile(beta=beta)

    threshold = sigma2_hat * (
        (1 + np.sqrt(gamma)) ** 2
        + t_tw
        * n ** (-2 / 3)
        * gamma ** (-1 / 6)
        * (1 + np.sqrt(gamma)) ** (4 / 3)
    )

    K_hat = int(np.sum(evals_sorted > threshold))

    return {
        "K_hat": K_hat,
        "sigma2_hat": sigma2_hat,
        "threshold": threshold,
        "gamma": gamma,
        "p_tilde": p_tilde,
        "k_indices": k_indices,
        "q_bulk": q_bulk,
        "evals_bulk": evals_bulk,
        "evals_sorted": evals_sorted,
        "tw_quantile": t_tw,
    }


def bema_algorithm1_from_data(Y, alpha=0.2, beta=0.1, center=False):
    """
    データ行列 Y から Algorithm 1 を実行する。

    Y は p x n、つまり
        p = 変数数
        n = サンプル数
    として扱う。

    sample covariance は S = Y Y^T / n。
    """
    Y = np.asarray(Y, dtype=float)

    if center:
        Y = Y - Y.mean(axis=1, keepdims=True)

    p, n = Y.shape

    if p <= n:
        S = Y @ Y.T / n
        evals = np.linalg.eigvalsh(S)
    else:
        # 非ゼロ固有値だけなら Y^T Y / n の固有値を使えばよい
        S_small = Y.T @ Y / n
        evals = np.linalg.eigvalsh(S_small)

    return bema_algorithm1_from_eigenvalues(
        evals=evals,
        p=p,
        n=n,
        alpha=alpha,
        beta=beta
    )

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.optimize as opt

def dyson_equalizer_algorithm1(Y):
    """
    Landa & Kluger (2024) - Algorithm 1: The Dyson Equalizer
    論文の数式と記法に完全に対応させた実装。

    Input:
        Y: Data matrix (m x n), m <= n
    Returns:
        Y_hat: Normalized data matrix
        x_hat: Row scaling vector
        y_hat: Column scaling vector
    """
    m, n = Y.shape
    if m > n:
        raise ValueError("Input matrix Y must have m <= n. Transpose Y if necessary.")

    # 1: Compute the SVD of Y
    # U: m x m, sigma: m, V_h: n x n
    U, sigma, V_h = np.linalg.svd(Y, full_matrices=True)
    V = V_h.T  # V \in R^{n x n} (右特異ベクトルを列に持つ行列)

    # 2: Set eta as the median singular value of Y
    eta = np.median(sigma)

    # 3: Compute the vectors g_hat^(1) and g_hat^(2)
    # 論文 (3) 式の計算（行列演算で高速化）
    term1 = eta / (sigma**2 + eta**2)
    term2 = term1 - (1 / eta)

    # U は m x m, sigma は要素数 m
    g1_hat = (U**2) @ term1

    # V は n x n. sum は k=1 から m までなので V の最初の m 列を使用
    g2_hat = (1 / eta) + (V[:, :m]**2) @ term2

    # 4: Compute the vectors x_hat and y_hat
    # L1ノルム ||g_hat^(1)||_1 と ||g_hat^(2)||_1 の計算
    g1_norm1 = np.sum(np.abs(g1_hat))
    g2_norm1 = np.sum(np.abs(g2_hat))

    # 論文 (4) 式の計算
    x_hat = (1 / np.sqrt(m - eta * g1_norm1)) * ((1 / g1_hat) - eta)
    y_hat = (1 / np.sqrt(n - eta * g2_norm1)) * ((1 / g2_hat) - eta)

    # 数値的安定性のための安全策（負値の平方根エラー回避）
    x_hat = np.maximum(1e-12, x_hat)
    y_hat = np.maximum(1e-12, y_hat)

    # 5: Form the normalized data matrix Y_hat
    # Y_hat = (D_{x_hat})^{-1/2} Y (D_{y_hat})^{-1/2}
    Y_hat = Y / (np.sqrt(x_hat[:, None]) * np.sqrt(y_hat[None, :]))

    return Y_hat, x_hat, y_hat

# ==========================================
# 動作検証用シミュレーション
# ==========================================
def run_simulation():
    # 1. パラメータの設定
    m, n = 1000, 5000  # (m <= n)
    gamma = m / n      # Q = n/m の逆数

    print(f"行列サイズ: {m} x {n}")

    # 異分散（ヘテロスケダスティック）構造の生成
    #np.random.seed(42)
    # Sigma_diag = np.random.uniform(0.1, 5.0, m)
    # Omega_diag = np.random.uniform(0.1, 5.0, n)

    Sigma_diag = np.random.exponential(1, m)
    Omega_diag = np.random.exponential(1, n)


    # Y ~ N(0, Omega \otimes Sigma) の生成
    Z = np.random.randn(m, n)
    Y = Z * np.sqrt(Sigma_diag[:, None]) * np.sqrt(Omega_diag[None, :])

    print("オリジナルの固有値を計算中...")
    X_orig = Y @ Y.T / n
    evals_orig = np.linalg.eigvalsh(X_orig)

    # ==========================================
    # Algorithm 1 の適用
    # ==========================================
    print("Algorithm 1 (Dyson Equalizer) を適用中...")
    Y_hat, x_hat, y_hat = dyson_equalizer_algorithm1(Y)

    # 補正後のESD計算
    X_hat = Y_hat @ Y_hat.T / n
    evals_hat = np.linalg.eigvalsh(X_hat)

    # ---------------------------------------------------------
    # MP分布の理論曲線の計算
    # ---------------------------------------------------------
    lambda_plus = (1 + np.sqrt(gamma))**2
    lambda_minus = (1 - np.sqrt(gamma))**2
    x_ax = np.linspace(max(0, lambda_minus - 0.5), lambda_plus + 0.5, 1000)

    valid_mask = (x_ax >= lambda_minus) & (x_ax <= lambda_plus)
    rho_mp = np.zeros_like(x_ax)
    rho_mp[valid_mask] = np.sqrt((lambda_plus - x_ax[valid_mask]) * (x_ax[valid_mask] - lambda_minus)) / (2 * np.pi * gamma * x_ax[valid_mask])

    result_orig = bema_algorithm1_from_data(Y, alpha=0.2, beta=0.1)

    print("BEMA Algorithm 1")
    print(f"sigma^2_hat = {result_orig['sigma2_hat']:.4f}")
    print(f"K_hat        = {result_orig['K_hat']}")
    print(f"threshold    = {result_orig['threshold']:.4f}")


    result_hat = bema_algorithm1_from_data(Y_hat, alpha=0.2, beta=0.1)

    print("After Dyson Equalizer")
    print(f"sigma^2_hat = {result_hat['sigma2_hat']:.4f}")
    print(f"K_hat        = {result_hat['K_hat']}")
    print(f"threshold    = {result_hat['threshold']:.4f}")

    sigma2_bema = result_hat['sigma2_hat']
    lambda_plus_bema = sigma2_bema * (1 + np.sqrt(gamma))**2
    lambda_minus = sigma2_bema * (1 - np.sqrt(gamma))**2



    # BEMAが推定したMP曲線の描画
    lambda_minus = sigma2_bema * (1 - np.sqrt(gamma))**2
    x = np.linspace(lambda_minus, lambda_plus_bema, 500)
    rho_mp_bema = np.sqrt((lambda_plus_bema - x) * (x - lambda_minus)) / (2 * np.pi * gamma * x * sigma2_bema)


    # ---------------------------------------------------------
    # プロット
    # ---------------------------------------------------------
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    bins = 100

    axes[0].hist(evals_orig, bins=bins, density=True, color='salmon', alpha=0.8)
    axes[0].plot(x, rho_mp_bema, 'k--', lw=2, label='Theoretical MP Law')
    axes[0].set_title('Original Heteroscedastic ESD', fontsize=14)
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    axes[1].hist(evals_hat, bins=bins, density=True, color='dodgerblue', alpha=0.8)
    axes[1].plot(x_ax, rho_mp, 'k--', lw=2, label='Theoretical MP Law')
    axes[1].set_title('Algorithm 1: Dyson Equalizer Corrected ESD', fontsize=14)
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()
    print("完了しました。")

if __name__ == "__main__":
    run_simulation()

基本的にこれでOK

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def dyson_equalizer_algorithm1(Y):
    """
    Landa & Kluger (2024) - Algorithm 1: The Dyson Equalizer
    論文の数式と記法に完全に対応させた実装。

    Input:
        Y: Data matrix (m x n), m <= n
    Returns:
        Y_hat: Normalized data matrix
        x_hat: Row scaling vector
        y_hat: Column scaling vector
    """
    m, n = Y.shape
    if m > n:
        raise ValueError("Input matrix Y must have m <= n. Transpose Y if necessary.")

    # 1: Compute the SVD of Y
    # U: m x m, sigma: m, V_h: n x n
    U, sigma, V_h = np.linalg.svd(Y, full_matrices=True)
    V = V_h.T  # V \in R^{n x n} (右特異ベクトルを列に持つ行列)

    # 2: Set eta as the median singular value of Y
    eta = np.median(sigma)

    # 3: Compute the vectors g_hat^(1) and g_hat^(2)
    # 論文 (3) 式の計算（行列演算で高速化）
    term1 = eta / (sigma**2 + eta**2)
    term2 = term1 - (1 / eta)

    # U は m x m, sigma は要素数 m
    g1_hat = (U**2) @ term1

    # V は n x n. sum は k=1 から m までなので V の最初の m 列を使用
    g2_hat = (1 / eta) + (V[:, :m]**2) @ term2

    # 4: Compute the vectors x_hat and y_hat
    # L1ノルム ||g_hat^(1)||_1 と ||g_hat^(2)||_1 の計算
    g1_norm1 = np.sum(np.abs(g1_hat))
    g2_norm1 = np.sum(np.abs(g2_hat))

    # 論文 (4) 式の計算
    x_hat = (1 / np.sqrt(m - eta * g1_norm1)) * ((1 / g1_hat) - eta)
    y_hat = (1 / np.sqrt(n - eta * g2_norm1)) * ((1 / g2_hat) - eta)

    # 数値的安定性のための安全策（負値の平方根エラー回避）
    x_hat = np.maximum(1e-12, x_hat)
    y_hat = np.maximum(1e-12, y_hat)

    # 5: Form the normalized data matrix Y_hat
    # Y_hat = (D_{x_hat})^{-1/2} Y (D_{y_hat})^{-1/2}
    Y_hat = Y / (np.sqrt(x_hat[:, None]) * np.sqrt(y_hat[None, :]))

    return Y_hat, x_hat, y_hat

# ==========================================
# 動作検証用シミュレーション
# ==========================================
def run_simulation():
    # 1. パラメータの設定
    m, n = 200, 500  # (m <= n)
    gamma = m / n      # Q = n/m の逆数

    print(f"行列サイズ: {m} x {n}")

    # # 異分散（ヘテロスケダスティック）構造の生成
    # #np.random.seed(42)
    # Sigma_diag = np.random.uniform(0.1, 5.0, m)
    # Omega_diag = np.random.uniform(0.1, 5.0, n)

    # # Y ~ N(0, Omega \otimes Sigma) の生成
    # Z = np.random.randn(m, n)
    # Y = Z * np.sqrt(Sigma_diag[:, None]) * np.sqrt(Omega_diag[None, :])

    # ==========================================
    # Step 1: 任意の非対角成分を持つ共分散行列を作成
    # ==========================================
    # 共分散行列は「正定値対称行列（Positive Definite Symmetric Matrix）」である必要があります。
    # ランダムな行列 A を用いて A @ A^T とすることで、数学的に正しい共分散行列を作成します。
    A_sigma = np.random.randn(m, n) * 0.1
    Sigma = A_sigma @ A_sigma.T + np.eye(m) * 1e-3  # 安定化のための微小な対角成分を追加

    A_omega = np.random.randn(n, n) * 0.1
    Omega = A_omega @ A_omega.T + np.eye(n) * 1e-3

    # ==========================================
    # Step 2: ベースとなる標準正規行列 Z の生成
    # ==========================================
    Z = np.random.randn(m, n)

    # ==========================================
    # Step 3: 共分散行列の「平方根（コレスキー分解）」を計算
    # ==========================================
    # Σ = L_sigma @ L_sigma^T を満たす下三角行列 L を取得します
    L_sigma = np.linalg.cholesky(Sigma)
    L_omega = np.linalg.cholesky(Omega)

    # ==========================================
    # Step 4: 一般の行列変量正規分布 Y の生成
    # ==========================================
    # 数式 Y = Σ^{1/2} Z Ω^{1/2} と等価な計算を、コレスキー因子を用いて行います
    # L_omega は下三角行列なので、右から掛けるときは転置 (.T) します
    Y = L_sigma @ Z @ L_omega.T

    print("オリジナルの固有値を計算中...")
    X_orig = Y @ Y.T / n
    evals_orig = np.linalg.eigvalsh(X_orig)

    # ==========================================
    # Algorithm 1 の適用
    # ==========================================
    print("Algorithm 1 (Dyson Equalizer) を適用中...")
    Y_hat, x_hat, y_hat = dyson_equalizer_algorithm1(Y)

    # 補正後のESD計算
    X_hat = Y_hat @ Y_hat.T / n
    evals_hat = np.linalg.eigvalsh(X_hat)

    # ---------------------------------------------------------
    # MP分布の理論曲線の計算
    # ---------------------------------------------------------
    lambda_plus = (1 + np.sqrt(gamma))**2
    lambda_minus = (1 - np.sqrt(gamma))**2
    x_ax = np.linspace(max(0, lambda_minus - 0.5), lambda_plus + 0.5, 1000)

    valid_mask = (x_ax >= lambda_minus) & (x_ax <= lambda_plus)
    rho_mp = np.zeros_like(x_ax)
    rho_mp[valid_mask] = np.sqrt((lambda_plus - x_ax[valid_mask]) * (x_ax[valid_mask] - lambda_minus)) / (2 * np.pi * gamma * x_ax[valid_mask])

    # ---------------------------------------------------------
    # プロット
    # ---------------------------------------------------------
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    bins = 100

    axes[0].hist(evals_orig, bins=bins, density=True, color='salmon', alpha=0.8)
    axes[0].plot(x_ax, rho_mp, 'k--', lw=2, label='Theoretical MP Law')
    axes[0].set_title('Original Heteroscedastic ESD', fontsize=14)
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    axes[1].hist(evals_hat, bins=bins, density=True, color='dodgerblue', alpha=0.8)
    axes[1].plot(x_ax, rho_mp, 'k--', lw=2, label='Theoretical MP Law')
    axes[1].set_title('Algorithm 1: Dyson Equalizer Corrected ESD', fontsize=14)
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()
    print("完了しました。")

if __name__ == "__main__":
    run_simulation()